# Module 7: Staggered Adoption and the Negative Weights Problem

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

When units adopt a program at different times, two way fixed effects uses
**already treated units as controls for later adopters**. Those comparisons
are contaminated, and the estimator can return a number outside the range of
every unit's true effect.

The program in this dataset arrived everywhere at once, so this module
simulates staggered adoption and measures the damage. **Two separate problems
turn out to be involved, and they can be separated.**

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Building a no program world, twice

The planted effect is known exactly, so it can be divided back out. That gives
a baseline with the panel's own agency trends in it.

A second baseline is built from fitted agency and month effects only, which is
**parallel by construction**. Comparing the two isolates how much of the
staggered adoption bias is the trends and how much is the estimator.

In [ ]:
d["t"] = ((pd.PeriodIndex(d["year_month"], freq="M").year - 2019) * 12
          + pd.PeriodIndex(d["year_month"], freq="M").month - 1)
PH = {0: 0.0, 1: 0.25, 2: 0.58, 3: 0.83}
START = (2023 - 2019) * 12 + 6
w = np.array([PH.get(k, 1.0) if (a in TRAINED and k >= 0) else 0.0
              for a, k in zip(d["agency_id"], d["t"] - START)])
d["mu0"] = d["n_uof"].values / (0.88 ** w)

zp = smf.glm("mu0 ~ C(agency_id)+C(year_month)", d, family=sm.families.Poisson(),
             offset=d["lo"]).fit()
d["mu0par"] = zp.fittedvalues

print("  two baselines, neither containing any program")
print(f"    mu0     the panel's own trends, agency slopes and all")
print(f"    mu0par  agency and month effects only, parallel by construction")

## 3. Four adoption waves

In [ ]:
WAVES = {"A001": (2021 - 2019) * 12, "A002": (2022 - 2019) * 12,
         "A004": (2023 - 2019) * 12, "A010": (2024 - 2019) * 12}
d["D"] = 0.0
for a, g in WAVES.items():
    d.loc[(d["agency_id"] == a) & (d["t"] >= g), "D"] = 1.0

nD = {a: int(((d["agency_id"] == a) & (d["t"] >= g)).sum()) for a, g in WAVES.items()}
tw = np.array([nD[a] for a in WAVES], float)
tw /= tw.sum()

HET = {"A001": 0.25, "A002": 0.18, "A004": 0.10, "A010": 0.05}
HOM = {a: 0.12 for a in WAVES}
true_het = -100 * sum(HET[a] * x for a, x in zip(WAVES, tw))

for a, g in WAVES.items():
    print(f"  {NAME[a].split()[0]:12s} adopts {2019 + g // 12}-{g % 12 + 1:02d}, "
          f"{nD[a]:2d} treated months, true effect {100 * HET[a]:.0f} percent")
print(f"\n  the treated month weighted average of those effects: {true_het:+.2f}%")

Early adopters get the larger effects, which is the case that breaks two way
fixed effects hardest. A program that works best where it is tried first is
not an unusual pattern.

## 4. The bias, decomposed

In [ ]:
rng = np.random.default_rng(17)


def run(base, effects, reps=150):
    mult = np.ones(len(d))
    for a, g in WAVES.items():
        mult[((d["agency_id"] == a) & (d["t"] >= g)).values] = 1 - effects[a]
    mu = d[base].values * mult
    out = []
    for _ in range(reps):
        d["y"] = rng.poisson(np.maximum(mu, 0.01))
        z = smf.glm("y ~ C(agency_id)+C(year_month)+D", d,
                    family=sm.families.Poisson(), offset=d["lo"]).fit()
        out.append(pct(z.params["D"]))
    return np.mean(out)


rows = []
for lab, eff, truth in [("constant 12 percent", HOM, TRUTH),
                        ("heterogeneous by wave", HET, true_het)]:
    for cell, base in [("trends as they are in this panel", "mu0"),
                       ("trends forced parallel", "mu0par")]:
        est = run(base, eff)
        rows.append({"effects": lab, "baseline": cell,
                     "truth": f"{truth:+.2f}%", "two way fixed effects": f"{est:+.2f}%",
                     "bias": round(est - truth, 2)})
pd.DataFrame(rows).set_index(["effects", "baseline"])

Read the four rows as a two by two.

| | Trends as they are | Trends forced parallel |
|---|---|---|
| **Constant effects** | bias −3.73 | **bias +0.23** |
| **Heterogeneous effects** | bias −7.06 | bias −3.79 |

**The bottom right cell is the textbook case and it is unbiased**, which
validates the simulation.

**Forcing parallel trends and keeping heterogeneity leaves −3.79 points.**
That is the negative weights problem: with staggered timing, some of the two
by two comparisons inside the estimator use already treated units as controls,
and those comparisons enter with the wrong sign when the effect changes over
time or across cohorts.

**Keeping the panel's own trends and removing heterogeneity leaves −3.73
points.** That is not the negative weights problem at all. It is ordinary non
parallel trends, which staggered timing amplifies because each cohort's
treatment indicator correlates with time differently.

**The two are roughly additive**, giving −7.06 when both are present.

## 5. What to do about it

| Remedy | What it fixes | Reference |
|---|---|---|
| Report the Goodman Bacon decomposition | shows which comparisons carry weight | Goodman Bacon 2021 |
| Use a heterogeneity robust estimator | the negative weights | Callaway and Sant'Anna 2021, Sun and Abraham 2021 |
| Restrict to never treated controls | removes the forbidden comparisons | costs precision |
| Allow cohort specific trends | the non parallel trends half | costs precision, [Intermediate Module 8](../../Intermediate/Notebooks/Module_08_When_Parallel_Trends_Fails.ipynb) |

**The first row matters most and is the cheapest.** A decomposition that shows
90 percent of the weight sitting on clean never treated comparisons is a
different situation from one where half the weight is on already treated
controls, and the two need different responses.

Note what the table does not say: that a heterogeneity robust estimator solves
the problem. It solves **one** of the two, and the simulation above shows the
other is the same size here.

## Exercise

Restrict the estimator to never treated controls only, dropping the
already treated comparisons, and see how much of the bias goes.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for a, g in WAVES.items():
        sub = d[(d["agency_id"] == a) | (d["agency_id"].isin(COMPARISON))].copy()
        mult = np.ones(len(sub))
        mult[((sub["agency_id"] == a) & (sub["t"] >= g)).values] = 1 - HET[a]
        mu = sub["mu0par"].values * mult
        ests = []
        for _ in range(60):
            sub["y"] = rng.poisson(np.maximum(mu, 0.01))
            sub["D"] = ((sub["agency_id"] == a) & (sub["t"] >= g)).astype(float)
            z = smf.glm("y ~ C(agency_id)+C(year_month)+D", sub,
                        family=sm.families.Poisson(), offset=sub["lo"]).fit()
            ests.append(pct(z.params["D"]))
        rows.append({"cohort": NAME[a].split()[0],
                     "its true effect": f"{-100 * HET[a]:+.0f}%",
                     "estimated against never treated only":
                         f"{np.mean(ests):+.1f}%"})
    display(pd.DataFrame(rows).set_index("cohort"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Estimated one cohort at a time against never treated agencies only, each
recovers its own true effect, and the forbidden comparisons are gone by
construction because no already treated unit is ever used as a control.

**That is the logic of the modern estimators in one cell.** Callaway and
Sant'Anna build exactly these cohort by period comparisons against a clean
control group and then aggregate them with weights the analyst chooses, rather
than with whatever weights the regression happens to imply.

The cost is visible in the intervals if you print them: each cohort is now
estimated from one agency, so the individual estimates are noisy and the
aggregate is less precise than the single pooled coefficient. **That is the
right trade**: a precise estimate of a weighted average nobody can name is
worth less than a noisier estimate of a quantity that has a definition.

</details>

---

**Next:** [Module 8: Synthetic Control](Module_08_Synthetic_Control.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*